# Notebook 11 – Cross Validation Strategies

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.

We also use `CustomerID` (for grouping) and `InvoiceDate` (for time order) to demonstrate strategies that need them.

## Setup: Load & Prepare Data

In [1]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df.sort_values('InvoiceDate')   # important for time series split
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
groups = df['CustomerID']
model = LogisticRegression(max_iter=1000)

## 1. K-Fold Cross Validation
Splits data into K equal folds, trains on K-1, tests on the remaining fold, repeats K times. **Best for:** general-purpose data with no special structure (independent rows, no time order, no groups).

In [2]:
from sklearn.model_selection import KFold, cross_val_score
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf)
print("K-Fold scores:", scores.round(3), "| mean:", scores.mean().round(3))

K-Fold scores: [0.897 0.9   0.897 0.892 0.895] | mean: 0.896


## 2. Stratified K-Fold
Like K-Fold, but keeps the same class ratio in every fold. **Best for:** classification tasks, especially with **imbalanced classes** (like ours — ~90% UK vs ~10% non-UK).

In [3]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf)
print("Stratified K-Fold scores:", scores.round(3), "| mean:", scores.mean().round(3))

Stratified K-Fold scores: [0.893 0.897 0.895 0.897 0.897] | mean: 0.896


## 3. Repeated K-Fold
Runs K-Fold **multiple times** with different random splits, then averages everything. **Best for:** getting a more stable, reliable estimate when you have a smaller dataset and want to reduce the luck of any single split.

In [4]:
from sklearn.model_selection import RepeatedKFold
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
scores = cross_val_score(model, X, y, cv=rkf)
print("Repeated K-Fold — mean:", scores.mean().round(3), "| std:", scores.std().round(3), f"(n={len(scores)} runs)")

Repeated K-Fold — mean: 0.896 | std: 0.007 (n=15 runs)


## 4. Leave-One-Out (LOO)
Each fold tests on just **one row**. Very thorough but very slow. **Best for:** very small datasets, where every data point is precious and you can afford the computation.

In [5]:
from sklearn.model_selection import LeaveOneOut
X_small, y_small = X[:100], y[:100]   # small subset - LOO is too slow otherwise
loo = LeaveOneOut()
scores = cross_val_score(model, X_small, y_small, cv=loo)
print("LOO mean accuracy:", scores.mean().round(3), f"(n={len(scores)} folds)")

LOO mean accuracy: 0.93 (n=100 folds)


## 5. Group K-Fold
Ensures that **all rows belonging to the same group** (e.g., the same customer) stay together in the same fold, never split across train and test. **Best for:** data with repeated measurements per entity (e.g., multiple orders per customer) — prevents the model from "cheating" by seeing the same customer in both train and test.

In [6]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits=5)
scores = cross_val_score(model, X, y, cv=gkf, groups=groups)
print("Group K-Fold scores:", scores.round(3), "| mean:", scores.mean().round(3))

Group K-Fold scores: [0.922 0.905 0.923 0.847 0.885] | mean: 0.896


## 6. Time Series Split
Splits data so that **training always comes before testing** in time — no shuffling, no looking into the future. **Best for:** time-ordered data (like our transactions over time), where using future data to predict the past would be unrealistic ("data leakage").

In [7]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(model, X, y, cv=tscv)  # X, y are already sorted by InvoiceDate
print("Time Series Split scores:", scores.round(3), "| mean:", scores.mean().round(3))

Time Series Split scores: [0.914 0.892 0.884 0.884 0.902] | mean: 0.895
